# Module 099: Capstone Project — Full-Stack Application

> Phase 11: Capstone | Estimated time: 6 hours | Milestone Project: Yes

This notebook demonstrates the core components of building a full-stack Task Manager application with FastAPI, SQLAlchemy, and Jinja2.

## 1. Database Models with SQLAlchemy

Define the data models that represent our application entities.

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Text, Boolean, DateTime, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, Session
from datetime import datetime, timezone

Base = declarative_base()

class Category(Base):
    __tablename__ = "categories"
    
    id = Column(Integer, primary_key=True)
    name = Column(String(100), nullable=False, unique=True)
    description = Column(Text, nullable=True)
    
    tasks = relationship("Task", back_populates="category")
    
    def __repr__(self) -> str:
        return f"<Category(id={self.id}, name='{self.name}')>"

class Task(Base):
    __tablename__ = "tasks"
    
    id = Column(Integer, primary_key=True)
    title = Column(String(200), nullable=False)
    description = Column(Text, nullable=True)
    completed = Column(Boolean, default=False)
    created_at = Column(DateTime, default=lambda: datetime.now(timezone.utc))
    category_id = Column(Integer, ForeignKey("categories.id"), nullable=True)
    
    category = relationship("Category", back_populates="tasks")
    
    def __repr__(self) -> str:
        return f"<Task(id={self.id}, title='{self.title}', completed={self.completed})>"

# Create in-memory database for demonstration
engine = create_engine("sqlite:///:memory:", echo=True)
Base.metadata.create_all(engine)
print("Database tables created successfully!")

## 2. Pydantic Schemas for Validation

Define request and response schemas with automatic validation.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
from datetime import datetime

class TaskCreate(BaseModel):
    title: str = Field(..., min_length=1, max_length=200, description="Task title")
    description: Optional[str] = Field(None, max_length=2000)
    category_id: Optional[int] = None

class TaskResponse(BaseModel):
    id: int
    title: str
    description: Optional[str] = None
    completed: bool
    created_at: datetime
    category_id: Optional[int] = None
    
    model_config = {"from_attributes": True}

class CategoryCreate(BaseModel):
    name: str = Field(..., min_length=1, max_length=100)
    description: Optional[str] = None

class CategoryResponse(BaseModel):
    id: int
    name: str
    description: Optional[str] = None
    
    model_config = {"from_attributes": True}

print("Pydantic schemas defined!")
print(TaskCreate(title="Buy groceries", description="Milk, eggs, bread"))

## 3. Database Operations (CRUD)

Implement create, read, update, and delete operations.

In [ ]:
from sqlalchemy.orm import Session

def create_category(db: Session, name: str, description: str = None) -> Category:
    category = Category(name=name, description=description)
    db.add(category)
    db.commit()
    db.refresh(category)
    return category

def create_task(db: Session, title: str, description: str = None, category_id: int = None) -> Task:
    task = Task(title=title, description=description, category_id=category_id)
    db.add(task)
    db.commit()
    db.refresh(task)
    return task

def get_tasks(db: Session, completed: bool = None) -> list[Task]:
    query = db.query(Task)
    if completed is not None:
        query = query.filter(Task.completed == completed)
    return query.all()

def mark_task_complete(db: Session, task_id: int) -> Task | None:
    task = db.query(Task).filter(Task.id == task_id).first()
    if task:
        task.completed = True
        db.commit()
        db.refresh(task)
    return task

def delete_task(db: Session, task_id: int) -> bool:
    task = db.query(Task).filter(Task.id == task_id).first()
    if task:
        db.delete(task)
        db.commit()
        return True
    return False

# Test CRUD operations
with Session(engine) as session:
    cat = create_category(session, "Personal", "Personal errands and tasks")
    print(f"Created category: {cat}")
    
    task = create_task(session, "Buy groceries", "Milk, eggs, bread", cat.id)
    print(f"Created task: {task}")
    
    all_tasks = get_tasks(session)
    print(f"All tasks: {all_tasks}")
    
    mark_task_complete(session, task.id)
    print(f"Task after completion: {task}")

## 4. FastAPI Application Structure

See how a minimal FastAPI app ties everything together.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, status
from typing import List

app = FastAPI(title="Task Manager API", version="1.0.0")

def get_db():
    db = Session(engine)
    try:
        yield db
    finally:
        db.close()

@app.get("/api/tasks", response_model=List[TaskResponse])
def list_tasks(completed: Optional[bool] = None, db: Session = Depends(get_db)):
    tasks = get_tasks(db, completed)
    return tasks

@app.post("/api/tasks", response_model=TaskResponse, status_code=status.HTTP_201_CREATED)
def create_new_task(task_data: TaskCreate, db: Session = Depends(get_db)):
    task = create_task(db, task_data.title, task_data.description, task_data.category_id)
    return task

@app.put("/api/tasks/{task_id}/complete", response_model=TaskResponse)
def complete_task(task_id: int, db: Session = Depends(get_db)):
    task = mark_task_complete(db, task_id)
    if not task:
        raise HTTPException(status_code=404, detail="Task not found")
    return task

@app.delete("/api/tasks/{task_id}", status_code=status.HTTP_204_NO_CONTENT)
def remove_task(task_id: int, db: Session = Depends(get_db)):
    if not delete_task(db, task_id):
        raise HTTPException(status_code=404, detail="Task not found")

print("FastAPI application defined!")
print(f"Routes: {[route.path for route in app.routes]}")

## 5. Testing the API

Test the API endpoints using TestClient.

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# Create a category first
# Tasks endpoint
response = client.get("/api/tasks")
print(f"GET /api/tasks: {response.status_code}")
print(f"Response: {response.json()}")

# Create a task
response = client.post("/api/tasks", json={"title": "Learn FastAPI", "description": "Build a full-stack app"})
print(f"\nPOST /api/tasks: {response.status_code}")
task_data = response.json()
print(f"Created task: {task_data}")

# Complete the task
task_id = task_data["id"]
response = client.put(f"/api/tasks/{task_id}/complete")
print(f"\nPUT /api/tasks/{task_id}/complete: {response.status_code}")
print(f"Completed task: {response.json()}")

# Delete the task
response = client.delete(f"/api/tasks/{task_id}")
print(f"\nDELETE /api/tasks/{task_id}: {response.status_code}")

# Verify deletion
response = client.get("/api/tasks")
print(f"\nGET /api/tasks after delete: {response.json()}")

## Next Steps

This notebook showed the core components. Now build the full application:
1. Add Jinja2 templates for a web UI
2. Implement CSV data import
3. Add authentication
4. Add logging and error handling
5. Deploy with uvicorn

👉 Go to [Module 100: Where to Go Next](../module-100-where-to-go-next/README.md) for career guidance.